# T10 — 고CER 화자 전사 패스 (1단계)

**이건 학습이 아니다.** 문장 경계를 잡으려면 large-v3 단어 타임스탬프가 먼저 필요한데,
KEJ·DTH의 `02-03`/`02-04`는 한 번도 인식을 돌린 적이 없다. 이 노트북이 그걸 만든다.

| 화자 | 기준 자모 CER | 02-03 (등록/개발) | 02-04 (held-out 평가) |
| --- | --- | --- | --- |
| KEJ | **0.520** — 최고 | 19.7분 / 1,728자 | 16.2분 / 1,074자 |
| DTH | 0.168 | 30.0분 / 1,414자 | 40.0분 / 1,516자 |

**파일럿과 다른 점: 등록과 평가가 서로 다른 과제다.** B1 파일럿은 같은 과제·같은 녹음에서
나눴기 때문에 과제 간 전이를 못 쟀다. 여기서는 잰다.

## 드라이브 최상위에 올릴 것

**`t10_highcer` 폴더 하나만 올리면 된다.** 16kHz 변환본(`t10_16k/`)이 그 안에 들어 있다.
전체 0.20 GB. 드라이브 최상위에 `MyDrive/t10_highcer/` 가 되도록 둔다.

**런타임 → 런타임 유형 변경 → T4 GPU → 저장.** 106분 분량이라 T4에서 10~20분 걸린다.

## 1. 설치 — `설치 OK`와 `torchao 없음`이 둘 다 찍혀야 한다

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음 — 런타임 유형을 T4 GPU로" ; pip -q install faster-whisper rapidfuzz ; pip -q uninstall -y torchao ; python -c "import torch, faster_whisper; import importlib.util as u; print('설치 OK', torch.__version__, 'torchao 없음' if u.find_spec('torchao') is None else 'TORCHAO 남아있음 — 런타임 재시작 후 다시')"


## 2. 드라이브 연결

In [ ]:
from google.colab import drive; drive.mount('/content/drive')


## 3. 전사 (10~20분)

디코딩은 B0와 **똑같이** 맞춘다(beam 5, VAD on, min_silence 500ms, 단어 타임스탬프).
D9의 greedy는 시연 서버와 적응 평가 패스에 적용되는 것이고, 여기서 바꾸면 B0 숫자와
비교가 깨진다. 이 패스는 정렬용 타임스탬프가 목적이다.

파일마다 `jamo` 값이 찍힌다. **KEJ 02-03/02-04의 기준 CER이 여기서 처음 측정된다.**

### 3a. 진단 — 실행이 0초 만에 끝나면 여기부터

`!cd A && B` 는 A가 없으면 B를 **실행하지 않고 조용히 끝난다.** 아래 셀이 실제 경로를 찍는다.

In [ ]:
import os, glob
D="/content/drive/MyDrive"
print("MyDrive 최상위:", sorted(os.listdir(D))[:40])
P=os.path.join(D,"t10_highcer")
print("\nt10_highcer 있음?", os.path.isdir(P))
if os.path.isdir(P):
    print("  안의 파일:", sorted(os.listdir(P)))
    W=os.path.join(P,"t10_16k")
    print("  t10_16k 있음?", os.path.isdir(W))
    if os.path.isdir(W):
        ws=sorted(glob.glob(os.path.join(W,"*.wav")))
        print("  wav %d개" % len(ws))
        for f in ws: print("    %8.1f MB  %s" % (os.path.getsize(f)/1e6, os.path.basename(f)))
    for need in ("t10_transcribe.py","t10_manifest.json","b0_run.py"):
        print("  %-22s %s" % (need, "OK" if os.path.isfile(os.path.join(P,need)) else "*** 없음 ***"))
else:
    print("\n업로드가 최상위에 안 들어갔다. 위 목록에서 실제 이름을 확인할 것.")


### 3b. 전사 실행

In [ ]:
!cd "/content/drive/MyDrive/t10_highcer" && python t10_transcribe.py || echo "=== 위 오류를 확인할 것. 아무것도 안 찍혔으면 폴더 경로가 틀린 것이다 (3a 진단 셀) ==="


## 끝나면

`t10_highcer/results/` 를 통째로 내려받아 `sw_challenge/experiments/t10_highcer/results/` 에 넣는다.

- `hyp_large-v3.json` — 단어 타임스탬프 포함. **이게 다음 단계의 입력이다**
- `t10_b0_summary.json` — 파일별 기준 CER

파일마다 체크포인트를 쓰므로 중간에 끊겨도 거기까지는 남는다.

### 안 될 때

| 증상 | 조치 |
| --- | --- |
| `b0_run.py not found` | `t10_highcer` 폴더를 통째로 올렸는지 확인 |
| `missing ... t10_16k` | `t10_16k/` 가 업로드한 `t10_highcer` 안에 같이 올라갔는지 확인 |
| `GPU 없음` | 런타임 유형을 T4로. 무료 티어에서 안 잡히면 나중에 |
| `ImportError ... torchao` | 1번 셀이 지우도록 돼 있다. `torchao 없음`이 찍혔는지 확인 |